In [ ]:
# Notebook imports
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Notebook configuration
FEATURES = [
    "spread_pct", "vwap_deviation", "return_1h", "log_return", "high_low_range", "vol_6h", "vol_24h"
 ]
WINDOW_Z = 30
WINDOW_IQR = 30
ROLL_WINDOW = 20
CONTAMINATION = 0.01
N_ESTIMATORS = 200
N_NEIGHBORS = 20

COLORS = {
    "price": "#2563eb",
    "anomaly": "#dc2626",
    "normal": "#6b7280",
}

In [ ]:
# State check: requires ML outputs and model-ready dataframe
required = ["df_model", "FEATURES"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing) +
        ". Run 1.4-abz-ml-modeling.ipynb first in the same kernel."
    )

missing_feats = [c for c in FEATURES if c not in df_model.columns]
if missing_feats:
    raise RuntimeError("df_model missing feature columns: " + ", ".join(missing_feats))
if "quote_datetime" not in df_model.columns:
    raise RuntimeError("df_model missing 'quote_datetime'. Run feature engineering first.")
print("Prerequisites OK. Proceed with ensemble analysis.")


In [ ]:
# Ensemble Model (IF + LOF)

if "df_model" not in globals():
    raise ValueError("df_model not found. Run feature-engineering cell first.")

features = FEATURES
part = df_model.copy().sort_values("quote_datetime").reset_index(drop=True)
active_features = [c for c in features if c in part.columns]
window_z = WINDOW_Z
roll_window = ROLL_WINDOW
contamination = CONTAMINATION

for col in active_features:
    mu = part[col].rolling(window_z).mean()
    sd = part[col].rolling(window_z).std()
    part[f"{col}_z"] = (part[col] - mu) / (sd + 1e-8)

ml_features = []
for col in active_features:
    zc = f"{col}_z"
    if zc in part.columns:
        ml_features.extend([col, zc])

part = part.dropna(subset=ml_features).reset_index(drop=True)
X = part[ml_features].values
X_scaled = StandardScaler().fit_transform(X)

iforest = IsolationForest(n_estimators=N_ESTIMATORS, contamination=contamination, random_state=42, n_jobs=-1)
iforest.fit(X_scaled)
part["iforest_flag"] = (iforest.predict(X_scaled) == -1).astype(int)
part["iforest_score"] = -iforest.decision_function(X_scaled)

lof = LocalOutlierFactor(n_neighbors=N_NEIGHBORS, contamination=contamination, n_jobs=-1)
part["lof_flag"] = (lof.fit_predict(X_scaled) == -1).astype(int)
part["lof_score"] = -lof.negative_outlier_factor_

def safe_minmax(s):
    den = s.max() - s.min()
    if den == 0 or pd.isna(den):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / den

part["if_norm"] = safe_minmax(part["iforest_score"])
part["lof_norm"] = safe_minmax(part["lof_score"])
part["ensemble_score"] = (part["if_norm"] + part["lof_norm"]) / 2.0
ens_thr = part["ensemble_score"].quantile(1 - contamination)

part["ensemble_flag"] = (
    ((part["iforest_flag"] == 1) & (part["lof_flag"] == 1)) |
    (part["ensemble_score"] >= ens_thr)
).astype(int)

mu2 = part["ensemble_score"].rolling(roll_window).mean()
sd2 = part["ensemble_score"].rolling(roll_window).std()
part["ensemble_z20"] = (part["ensemble_score"] - mu2) / (sd2 + 1e-8)

flagged = part[part["ensemble_flag"] == 1]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=("Price with Outliers", "Ensemble Score")
)

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["close"],
        name="Close Price",
        line=dict(color=COLORS["normal"], width=1),
        opacity=0.5,
    ),
    row=1, col=1,
 )

fig.add_trace(
    go.Scatter(
        x=flagged["quote_datetime"],
        y=flagged["close"],
        mode="markers",
        name="Outliers",
        marker=dict(color=COLORS["anomaly"], size=6),
    ),
    row=1, col=1,
 )

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["ensemble_z20"],
        name="Ensemble Score",
        line=dict(color=COLORS["price"], width=0.8),
    ),
    row=2, col=1,
 )

fig.add_hline(y=3, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-3, line_dash="dash", line_color="red", row=2, col=1)

fig.update_layout(height=800, title_text="Ensemble Outlier Detection Dashboard", template="plotly_white", showlegend=True)
fig.show()

print("Ensemble Model anomalies:", int(part["ensemble_flag"].sum()))

Ensemble Model anomalies: 154


In [ ]:
# Combined Model (IQR + Z_score + IF + LOF)

if "df_model" not in globals():
    raise ValueError("df_model not found. Run feature-engineering cell first.")

features = FEATURES
part = df_model.copy().sort_values("quote_datetime").reset_index(drop=True)
active_features = [c for c in features if c in part.columns]
window_z = WINDOW_Z
window_iqr = WINDOW_IQR
roll_window = ROLL_WINDOW
contamination = CONTAMINATION

# Z part
for col in active_features:
    mu = part[col].rolling(window_z).mean()
    sd = part[col].rolling(window_z).std()
    part[f"{col}_z"] = (part[col] - mu) / (sd + 1e-8)

if "log_return_z" in part.columns:
    part["z_score"] = part["log_return_z"]
else:
    z_cols = [f"{c}_z" for c in active_features if f"{c}_z" in part.columns]
    part["z_score"] = part[z_cols].abs().max(axis=1)
part["z_flag"] = (part["z_score"].abs() >= 3).astype(int)

# IQR part
for col in active_features:
    q1 = part[col].rolling(window_iqr).quantile(0.25)
    q3 = part[col].rolling(window_iqr).quantile(0.75)
    iqr = q3 - q1
    part[f"{col}_iqr_flag"] = ((part[col] < (q1 - 1.5 * iqr)) | (part[col] > (q3 + 1.5 * iqr))).astype(int)

iqr_cols = [f"{c}_iqr_flag" for c in active_features if f"{c}_iqr_flag" in part.columns]
part["iqr_flag"] = part[iqr_cols].any(axis=1).astype(int)

# ML matrix
ml_features = []
for col in active_features:
    zc = f"{col}_z"
    if zc in part.columns:
        ml_features.extend([col, zc])

part = part.dropna(subset=ml_features).reset_index(drop=True)
X = part[ml_features].values
X_scaled = StandardScaler().fit_transform(X)

# IF + LOF
iforest = IsolationForest(n_estimators=N_ESTIMATORS, contamination=contamination, random_state=42, n_jobs=-1)
iforest.fit(X_scaled)
part["iforest_flag"] = (iforest.predict(X_scaled) == -1).astype(int)
part["iforest_score"] = -iforest.decision_function(X_scaled)

lof = LocalOutlierFactor(n_neighbors=N_NEIGHBORS, contamination=contamination, n_jobs=-1)
part["lof_flag"] = (lof.fit_predict(X_scaled) == -1).astype(int)
part["lof_score"] = -lof.negative_outlier_factor_

def safe_minmax(s):
    den = s.max() - s.min()
    if den == 0 or pd.isna(den):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / den

part["if_norm"] = safe_minmax(part["iforest_score"])
part["lof_norm"] = safe_minmax(part["lof_score"])
part["ensemble_score"] = (part["if_norm"] + part["lof_norm"]) / 2.0
ens_thr = part["ensemble_score"].quantile(1 - contamination)

part["ensemble_flag"] = (
    ((part["iforest_flag"] == 1) & (part["lof_flag"] == 1)) |
    (part["ensemble_score"] >= ens_thr)
).astype(int)

# Combined condition
part["combined_flag"] = (
    (part["iqr_flag"] == 1) &
    (part["z_flag"] == 1) &
    (part["ensemble_flag"] == 1)
).astype(int)

Combined anomalies: 83


In [203]:
# Safe min-max normalization
def safe_minmax(series):
    s = series.copy()
    den = s.max() - s.min()
    if den == 0 or pd.isna(den):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / den


part["z_norm"] = safe_minmax(part["z_score"].abs())
part["iqr_norm"] = safe_minmax(part["iqr_flag"].astype(float))
part["if_norm"] = safe_minmax(part["iforest_score"])
part["lof_norm"] = safe_minmax(part["lof_score"])

# Equal weighting (baseline)
part["baseline_anomaly_score"] = (
    0.25 * part["z_norm"] +
    0.25 * part["iqr_norm"] +
    0.25 * part["if_norm"] +
    0.25 * part["lof_norm"]
)

# Dynamic Variance-Based Weighting
score_matrix = part[["z_norm", "iqr_norm", "if_norm", "lof_norm"]]
weights = score_matrix.std()
weights = weights / weights.sum()

print("Dynamic Weights Used:")
print(weights)

part["weighted_final_score"] = (
    weights["z_norm"]   * part["z_norm"] +
    weights["iqr_norm"] * part["iqr_norm"] +
    weights["if_norm"]  * part["if_norm"] +
    weights["lof_norm"] * part["lof_norm"]
)

top10 = (
    part[part["combined_flag"] == 1]
    .sort_values("weighted_final_score", ascending=False)
    .head(10)
)

top10_output = top10[
    [
        "quote_datetime",
        "close",
        "baseline_anomaly_score",
        "weighted_final_score"
    ]
].reset_index(drop=True)

print("\nTop 10 Combined Anomalies:")
print(top10_output)

Dynamic Weights Used:
z_norm      0.176024
iqr_norm    0.617073
if_norm     0.180147
lof_norm    0.026755
dtype: float64

Top 10 Combined Anomalies:
       quote_datetime    close  baseline_anomaly_score  weighted_final_score
0 2026-01-29 10:30:00  426.330                0.815974              0.978059
1 2017-10-27 10:30:00   85.080                0.756427              0.964442
2 2025-05-01 10:30:00  431.135                0.767662              0.962769
3 2023-07-18 12:30:00  364.080                0.729099              0.947326
4 2023-04-26 10:30:00  295.320                0.727323              0.946224
5 2024-10-31 10:30:00  407.740                0.730364              0.945241
6 2022-10-26 10:30:00  233.640                0.731621              0.941015
7 2023-01-04 10:30:00  227.150                0.710464              0.937901
8 2021-10-27 10:30:00  321.260                0.708718              0.936714
9 2018-03-26 10:30:00   92.120                0.704665              0.933521


In [235]:
# PCA using feature_cols

# 1) Validating required inputs
if "feature_cols" not in globals():
    raise ValueError("feature_cols not found. Run the feature-engineering cell first.")

missing_cols = [c for c in feature_cols if c not in df_model.columns]
if missing_cols:
    raise ValueError(f"These feature_cols are missing in df_model: {missing_cols}")

# 2) Building PCA input
part_pca = df_model.dropna(subset=feature_cols).copy()
X = part_pca[feature_cols].values
X_scaled = StandardScaler().fit_transform(X)

# 3) Fitting PCA (2 components)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 4) Writing pca1/pca2 back to df_model
df_model["pca1"] = np.nan
df_model["pca2"] = np.nan
df_model.loc[part_pca.index, "pca1"] = X_pca[:, 0]
df_model.loc[part_pca.index, "pca2"] = X_pca[:, 1]

# 5) Loadings and dominant variables
input_features = feature_cols
loading_df = pd.DataFrame(
    pca.components_.T,
    index=input_features,
    columns=["pca1_weight", "pca2_weight"]
)

variable_1 = loading_df["pca1_weight"].abs().idxmax()  # dominant for pca1
variable_2 = loading_df["pca2_weight"].abs().idxmax()  # dominant for pca2

# 6) Equations
def make_equation(pc_idx, pc_name):
    terms = [f"({w:+.6f})*{f}" for f, w in zip(input_features, pca.components_[pc_idx])]
    return f"{pc_name} = " + " ".join(terms)

eq_pca1 = make_equation(0, "pca1")
eq_pca2 = make_equation(1, "pca2")

# 7) Output
print("PCA components successfully added to df_model.")


x_label = f"Variable 1 (pca1): {variable_1} (weight = {loading_df.loc[variable_1, 'pca1_weight']:+.6f})"
y_label = f"Variable 2 (pca2): {variable_2} (weight = {loading_df.loc[variable_2, 'pca2_weight']:+.6f})"
print("\nAxis labels:")
print(x_label)
print(y_label)

PCA components successfully added to df_model.

Axis labels:
Variable 1 (pca1): vol_24h (weight = +0.586848)
Variable 2 (pca2): log_return (weight = +0.719677)


In [ ]:
# Prepare Data
corr_features = [
    "spread_pct",
    "vwap_deviation",
    "return_1h",
    "log_return",
    "high_low_range",
    "vol_6h",
    "vol_24h",
    "pca1",
    "pca2"
]

# Keep only existing columns
corr_features = [c for c in corr_features if c in df_model.columns]

# Compute correlation matrix
corr_matrix = df_model[corr_features].corr().round(2)

# Heatmap
fig = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale="RdBu_r",   # Blue → White → Red
    showscale=True
)

# Update layout
fig.update_layout(
    title_text="Correlation Matrix (Features + PCA Components)",
    template="plotly_white"
)

fig.show()

In [ ]:
if "is_anomaly" not in df_model.columns:
    df_model["is_anomaly"] = 0
    if "part" in globals() and "combined_flag" in part.columns:
        df_model.loc[part.index, "is_anomaly"] = part["combined_flag"].values

if "anomaly_score" not in df_model.columns:
    df_model["anomaly_score"] = 0.0
    if "part" in globals() and "ensemble_score" in part.columns:
        df_model.loc[part.index, "anomaly_score"] = part["ensemble_score"].values

threshold = df_model["anomaly_score"].quantile(0.99)
print(f"Anomaly threshold (99th percentile): {threshold:.4f}")


In [ ]:
# PCA scatter plot
fig = go.Figure()

# Add Normal points - lighter, muted color
fig.add_trace(go.Scatter(
    x=df_model[df_model['is_anomaly'] == False]['pca1'],
    y=df_model[df_model['is_anomaly'] == False]['pca2'],
    mode='markers',
    name='Normal',
    marker=dict(color='#cbd5e1', size=6, opacity=0.4), # Muted slate color
    hovertemplate='PCA1: %{x:.2f}<br>PCA2: %{y:.2f}<extra></extra>'
))

# Anomaly points - darker, saturated color
fig.add_trace(go.Scatter(
    x=df_model[df_model['is_anomaly'] == True]['pca1'],
    y=df_model[df_model['is_anomaly'] == True]['pca2'],
    mode='markers',
    name='Anomaly',
    marker=dict(color='#1e293b', size=8, symbol='diamond', opacity=1.0), # Dark slate/navy color
    hovertemplate='<b>Anomaly</b><br>PCA1: %{x:.2f}<br>PCA2: %{y:.2f}<extra></extra>'
))

fig.update_layout(
    title='Cluster Separation in PCA Space: Anomaly Contrast (Interactive)',
    xaxis_title='PCA Component 1 (vol_24h)',
    yaxis_title='PCA Component 2 (log_return)',
    template='plotly_white',
    hovermode='closest',
    height=600,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

In [243]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['quote_datetime'], y=df['spread_pct'], name='Spread %', line=dict(color='#059669', width=0.8)))
fig.update_layout(title='Bid-Ask Spread % Over Time', xaxis_title='Date', yaxis_title='Spread (%)', template='plotly_white')
fig.show()

### Relationship between Anomaly Score and Accuracy

In Anomaly Detection, we don't usually use 'Accuracy' because the classes are highly imbalanced (99% normal, 1% anomaly). If a model simply said 'nothing is an anomaly,' it would have 99% accuracy but be useless.

Instead, we look at:
- **Precision**: Of the points flagged by the score, how many were actually market shocks?
- **Recall**: Of all market shocks that happened, how many did the score successfully catch?

**The Conversion Logic:**
`Anomaly Flag = 1 if Anomaly Score > Threshold else 0`

In [ ]:
# Visualize the 'Confidence' of our scores
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df_model['anomaly_score'],
    nbinsx=50,
    marker_color=COLORS['price'],
    opacity=0.7,
    name='Score Distribution'
))

fig.add_vline(x=threshold, line_dash="dash", line_color=COLORS['anomaly'],
              annotation_text=f'Decision Threshold ({threshold:.2f})')

fig.update_layout(
    title='Distribution of Ensemble Anomaly Scores',
    xaxis_title='Anomaly Score (Probability of Outlierness)',
    yaxis_title='Frequency',
    template='plotly_white',
    showlegend=False
)
fig.show()

---
## 14. Summary of Key Findings

### Dataset
- **13,617 rows** of hourly MSFT data spanning **~9 years** (Jan 2017 - Mar 2026)
- **14 columns**: OHLCV + bid/ask/mid/vwap + metadata
- Data frequency: primarily 1-hour bars with half-hour closing bars at 16:00

### Data Quality
- **3 rows** with bid > ask (data inconsistencies to flag)
- **2 rows** with extreme single-bar returns (>20%) -- likely stock split events
- No negative prices, no duplicate timestamps
- OHLC consistency: all rows pass high >= low and close in [low, high]

### Seasonality
- Highest volatility at market open (10:30 AM) and close (15:30-16:00)
- Monday and Friday tend to show slightly different return patterns

### Anomaly Definition (Task Section 4)
Based on EDA findings, we define anomalies as data points where:
1. **Price returns** deviate significantly from historical norms (e.g., >3 sigma)
2. **Bid-ask spread** is abnormally wide (indicating market stress / low liquidity)
3. **VWAP deviation** is extreme (unusual buying/selling pressure)
4. **Multi-feature outlier**: combination of price, spread, and VWAP signals
   flagged by unsupervised ML models (IsolationForest, LOF)